# 03 — Preprocessing Dataset TMDB Menggunakan PySpark

Notebook ini digunakan untuk melakukan tahap *preprocessing* setelah proses scraping, enrichment, dan EDA.  
Input utama notebook ini adalah file:

```text
tmdb_movies_with_eda_features.csv
```

Output utama notebook ini adalah dataset yang lebih siap digunakan untuk:

1. model rekomendasi berbasis konten,
2. pembuatan dokumen film untuk embedding,
3. penyimpanan metadata ke MongoDB,
4. integrasi semantic search ke Qdrant,
5. analisis lanjutan menggunakan Spark ML atau Python ML.

Notebook ini tetap menggunakan **PySpark** sebagai mesin pemrosesan utama.  
Khusus untuk proses export akhir, notebook menggunakan **Pandas CSV export** agar aman di Windows, karena Spark lokal di Windows kadang gagal menulis file Parquet/CSV akibat kebutuhan native Hadoop library.

## 1. Setup Environment dan SparkSession

Bagian ini menyiapkan konfigurasi dasar PySpark agar bisa berjalan stabil di Windows + VSCode.  
Konfigurasi penting yang dilakukan:

- memastikan `JAVA_HOME` memakai JDK 17,
- membuat folder temporary Spark yang aman,
- memastikan Python driver dan executor memakai `.venv` yang sama,
- menjalankan Spark dalam mode lokal.

Jika notebook sudah berhasil berjalan, kamu boleh menaikkan `local[1]` menjadi `local[2]` atau `local[*]`, tetapi untuk awal gunakan konfigurasi yang paling stabil dulu.

In [13]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import *
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml import Pipeline

# Setup Java untuk Windows
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Temp folder aman untuk Spark di Windows
Path("C:/tmp/spark-temp").mkdir(parents=True, exist_ok=True)
os.environ["TEMP"] = "C:/tmp/spark-temp"
os.environ["TMP"] = "C:/tmp/spark-temp"

# Pastikan PySpark memakai Python dari venv yang sedang aktif
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("TMDB Movie Recommendation - Preprocessing")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Python executable:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Spark version: 4.1.2
Python executable: d:\Semester 6\BigData\movie-recommendation-bigdata\.venv\Scripts\python.exe
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot


## 2. Menentukan Lokasi File Input dan Output

Notebook ini dibuat fleksibel agar bisa langsung digunakan di struktur project kamu.  
Notebook akan mencari file input pada beberapa kemungkinan lokasi, misalnya:

- `data/processed/tmdb/tmdb_movies_with_eda_features.csv`
- `data/raw/tmdb/tmdb_movies_with_eda_features.csv`
- `reports/eda_outputs/tmdb_movies_with_eda_features.csv`
- file di root project

Jika file kamu berada di lokasi lain, ubah variabel `DATA_PATH` secara manual.

In [14]:
from pathlib import Path

# Ambil current working directory
CURRENT_DIR = Path.cwd()

# Jika notebook dijalankan dari folder notebooks, naik satu level ke root project
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Folder hasil EDA
EDA_DIR = PROJECT_ROOT / "data" / "interim" / "eda"

# File input hasil EDA
DATA_PATH = EDA_DIR / "tmdb_movies_with_eda_features.csv"

# Folder output preprocessing
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "tmdb"
REPORT_DIR = PROJECT_ROOT / "reports" / "preprocessing"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Output utama untuk sistem rekomendasi
PREPROCESSED_FULL_OUTPUT = PROCESSED_DIR / "tmdb_movies_preprocessed_full.csv"
MODEL_READY_OUTPUT = PROCESSED_DIR / "tmdb_movies_model_ready.csv"
MOVIE_DOCUMENTS_OUTPUT = PROCESSED_DIR / "tmdb_movie_documents.csv"
PREPROCESSING_SUMMARY_OUTPUT = REPORT_DIR / "preprocessing_summary.csv"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("EDA folder:", EDA_DIR)
print("Data path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())
print("Processed output:", PROCESSED_DIR)
print("Report output:", REPORT_DIR)

Current dir: d:\Semester 6\BigData\movie-recommendation-bigdata\notebooks
Project root: d:\Semester 6\BigData\movie-recommendation-bigdata
EDA folder: d:\Semester 6\BigData\movie-recommendation-bigdata\data\interim\eda
Data path: d:\Semester 6\BigData\movie-recommendation-bigdata\data\interim\eda\tmdb_movies_with_eda_features.csv
File exists: True
Processed output: d:\Semester 6\BigData\movie-recommendation-bigdata\data\processed\tmdb
Report output: d:\Semester 6\BigData\movie-recommendation-bigdata\reports\preprocessing


## 3. Membaca Dataset Menggunakan PySpark

Pada tahap ini file CSV dibaca dengan PySpark.  
Beberapa opsi penting:

- `header=True` agar baris pertama dibaca sebagai nama kolom,
- `inferSchema=True` agar Spark menebak tipe data,
- `multiLine=True` untuk mengantisipasi teks panjang yang mengandung baris baru,
- `quote` dan `escape` untuk menjaga isi teks seperti overview, cast, dan keywords tetap aman.

In [15]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(str(DATA_PATH))
)

print("Jumlah baris awal :", df_raw.count())
print("Jumlah kolom awal :", len(df_raw.columns))

df_raw.printSchema()
df_raw.show(5, truncate=False)

Jumlah baris awal : 80713
Jumlah kolom awal : 71
root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- genre_ids: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- adult: boolean (nullable = true)
 |-- video: boolean (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- source_config: string (nullable = true)
 |-- source_start_date: date (nullable = true)
 |-- source_end_date: date (nullable = true)
 |-- source_page: integer (nullable = true)
 |-- collected_at: timestamp (nullable = true)
 |-- runtime: double (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string

## 4. Standarisasi Nama Kolom

Tahap ini memastikan nama kolom konsisten dan aman digunakan pada proses PySpark.  
Jika ada spasi atau karakter yang kurang aman pada nama kolom, akan diganti menjadi format `snake_case`.

In [16]:
import re

def to_snake_case(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.lower().strip("_")

rename_map = {c: to_snake_case(c) for c in df_raw.columns}

df = df_raw
for old_col, new_col in rename_map.items():
    if old_col != new_col:
        df = df.withColumnRenamed(old_col, new_col)

print("Contoh kolom setelah standarisasi:")
print(df.columns[:30])

Contoh kolom setelah standarisasi:
['id', 'title', 'original_title', 'original_language', 'overview', 'release_date', 'genre_ids', 'genres', 'popularity', 'vote_average', 'vote_count', 'adult', 'video', 'poster_path', 'backdrop_path', 'source_config', 'source_start_date', 'source_end_date', 'source_page', 'collected_at', 'runtime', 'status', 'tagline', 'budget', 'revenue', 'imdb_id', 'homepage', 'genres_detail', 'production_companies', 'production_countries']


## 5. Validasi Kolom Penting

Sebelum preprocessing dilanjutkan, notebook mengecek apakah kolom-kolom penting tersedia.  
Kolom penting untuk sistem rekomendasi film meliputi:

- identitas film: `id`, `title`,
- metadata teks: `overview`, `genres`, `director`, `top_cast`, `keywords`,
- metadata numerik: `popularity`, `vote_average`, `vote_count`, `runtime`,
- informasi tanggal: `release_date`, `release_year`.

In [17]:
required_columns = [
    "id", "title", "overview", "genres", "popularity", "vote_average",
    "vote_count", "release_date", "runtime", "director", "top_cast", "keywords"
]

missing_required = [c for c in required_columns if c not in df.columns]

if missing_required:
    print("Kolom penting yang belum ada:", missing_required)
else:
    print("Semua kolom penting tersedia.")

Semua kolom penting tersedia.


## 6. Casting Tipe Data

Tahap ini mengubah tipe data agar lebih konsisten.  
Beberapa contoh:

- `id` menjadi integer/long,
- `budget`, `revenue`, `popularity`, `vote_average`, `runtime` menjadi numerik,
- `release_date` menjadi format tanggal,
- kolom flag seperti `adult` dan `video` menjadi boolean/string sesuai kebutuhan.

Casting ini penting agar agregasi, filtering, dan feature engineering tidak gagal akibat tipe data yang tidak konsisten.

In [18]:
# Helper untuk casting hanya jika kolom tersedia
def cast_if_exists(dataframe, col_name, dtype):
    if col_name in dataframe.columns:
        return dataframe.withColumn(col_name, F.col(col_name).cast(dtype))
    return dataframe

numeric_double_cols = [
    "popularity", "vote_average", "runtime", "budget", "revenue",
    "profit", "roi", "metadata_completeness_score"
]

numeric_int_cols = [
    "id", "vote_count", "release_year", "release_month", "release_decade",
    "source_page", "overview_length", "tagline_length", "title_length",
    "genres_detail_count", "production_companies_count", "production_countries_count",
    "spoken_languages_count", "writers_count", "top_cast_count", "keywords_count",
    "has_budget", "has_revenue", "has_budget_revenue", "has_overview",
    "has_tagline", "has_homepage", "has_poster", "has_backdrop",
    "has_genre", "has_director", "has_top_cast", "has_keywords",
    "has_runtime_valid", "has_vote_count_valid", "has_release_date"
]

for c in numeric_double_cols:
    df = cast_if_exists(df, c, T.DoubleType())

for c in numeric_int_cols:
    df = cast_if_exists(df, c, T.IntegerType())

date_cols = ["release_date", "source_start_date", "source_end_date", "collected_at", "enriched_at"]
for c in date_cols:
    if c in df.columns:
        if c in ["collected_at", "enriched_at"]:
            df = df.withColumn(c + "_ts", F.to_timestamp(F.col(c)))
        else:
            df = df.withColumn(c + "_parsed_clean", F.to_date(F.col(c)))

df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- genre_ids: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- adult: boolean (nullable = true)
 |-- video: boolean (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- source_config: string (nullable = true)
 |-- source_start_date: date (nullable = true)
 |-- source_end_date: date (nullable = true)
 |-- source_page: integer (nullable = true)
 |-- collected_at: timestamp (nullable = true)
 |-- runtime: double (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- budget: double (nullable 

## 7. Deduplikasi Data

Karena data berasal dari scraping bertahap, kemungkinan film yang sama muncul lebih dari satu kali tetap harus dicek.  
Deduplikasi dilakukan berdasarkan kolom `id` karena `id` TMDB adalah identitas unik film.

Strategi yang digunakan:

1. hitung jumlah duplikat,
2. urutkan data berdasarkan kelengkapan metadata,
3. simpan satu baris terbaik untuk setiap `id`.

In [19]:
initial_rows = df.count()
distinct_id_rows = df.select("id").distinct().count()

print("Rows awal       :", initial_rows)
print("Distinct id     :", distinct_id_rows)
print("Duplicate by id :", initial_rows - distinct_id_rows)

# Jika ada metadata_completeness_score, gunakan itu untuk memilih baris terbaik
order_cols = []
if "metadata_completeness_score" in df.columns:
    order_cols.append(F.col("metadata_completeness_score").desc_nulls_last())
if "vote_count" in df.columns:
    order_cols.append(F.col("vote_count").desc_nulls_last())

if order_cols:
    window_spec = __import__("pyspark.sql.window").sql.window.Window.partitionBy("id").orderBy(*order_cols)
    df_dedup = (
        df
        .withColumn("_rn", F.row_number().over(window_spec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )
else:
    df_dedup = df.dropDuplicates(["id"])

print("Rows setelah deduplikasi:", df_dedup.count())

Rows awal       : 80713
Distinct id     : 80713
Duplicate by id : 0
Rows setelah deduplikasi: 80713


## 8. Filtering Data Dasar

Tidak semua baris layak digunakan untuk model rekomendasi.  
Tahap ini membuang data yang sangat bermasalah, misalnya:

- tidak memiliki `id`,
- tidak memiliki `title`,
- bukan film rilis valid,
- vote count negatif atau rating di luar rentang.

Filtering ini tidak dibuat terlalu ketat agar jumlah data tetap besar.

In [20]:
df_clean = df_dedup

# Wajib punya id dan title
df_clean = df_clean.filter(F.col("id").isNotNull())
df_clean = df_clean.filter(F.col("title").isNotNull() & (F.length(F.trim(F.col("title"))) > 0))

# Adult/video biasanya tidak digunakan untuk rekomendasi film umum
if "adult" in df_clean.columns:
    df_clean = df_clean.filter((F.col("adult") == False) | F.col("adult").isNull())

# Rating dan vote count dibuat aman
if "vote_average" in df_clean.columns:
    df_clean = df_clean.withColumn(
        "vote_average",
        F.when((F.col("vote_average") < 0) | (F.col("vote_average") > 10), None)
         .otherwise(F.col("vote_average"))
    )

if "vote_count" in df_clean.columns:
    df_clean = df_clean.withColumn(
        "vote_count",
        F.when(F.col("vote_count") < 0, 0).otherwise(F.col("vote_count"))
    )

print("Rows setelah filtering dasar:", df_clean.count())

Rows setelah filtering dasar: 80712


## 9. Penanganan Missing Value

Tahap ini tidak sekadar menghapus missing value.  
Untuk sistem rekomendasi, beberapa kolom teks tetap bisa digunakan walaupun kosong, sehingga nilainya diisi dengan string kosong.

Strategi:

- teks kosong diisi `""`,
- numerik penting diisi dengan nilai aman,
- flag tambahan dibuat agar model/analisis tetap tahu apakah data aslinya tersedia atau tidak.

In [21]:
text_cols = [
    "title", "original_title", "original_language", "overview", "genres",
    "tagline", "genres_detail", "production_companies", "production_countries",
    "spoken_languages", "director", "writers", "top_cast", "keywords",
    "status", "imdb_id", "homepage"
]

# Pastikan kolom teks penting tetap ada, walaupun di CSV awal tidak tersedia.
# Ini membuat cell berikutnya tidak mudah error ketika dataset punya variasi kolom.
for c in text_cols:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c, F.coalesce(F.col(c).cast("string"), F.lit("")))
    else:
        df_clean = df_clean.withColumn(c, F.lit(""))

genre_source = F.coalesce(F.col("genres_detail"), F.col("genres"))

# Flag ketersediaan data untuk menilai kualitas metadata film
df_clean = (
    df_clean
    .withColumn("is_overview_available", F.when(F.length(F.trim(F.col("overview"))) > 0, 1).otherwise(0))
    .withColumn("is_genre_available", F.when(F.length(F.trim(genre_source)) > 0, 1).otherwise(0))
    .withColumn("is_director_available", F.when(F.length(F.trim(F.col("director"))) > 0, 1).otherwise(0))
    .withColumn("is_cast_available", F.when(F.length(F.trim(F.col("top_cast"))) > 0, 1).otherwise(0))
    .withColumn("is_keyword_available", F.when(F.length(F.trim(F.col("keywords"))) > 0, 1).otherwise(0))
)

# Nilai numerik aman
safe_numeric_defaults = {
    "popularity": 0.0,
    "vote_average": 0.0,
    "vote_count": 0,
    "runtime": 0.0,
    "budget": 0.0,
    "revenue": 0.0,
    "profit": 0.0,
    "roi": 0.0,
    "metadata_completeness_score": 0.0,
}

for c, default_value in safe_numeric_defaults.items():
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c, F.coalesce(F.col(c), F.lit(default_value)))
    else:
        df_clean = df_clean.withColumn(c, F.lit(default_value))

df_clean.select(
    "id", "title", "is_overview_available", "is_genre_available",
    "is_director_available", "is_cast_available", "is_keyword_available"
).show(5, truncate=60)

+---+-------------------+---------------------+------------------+---------------------+-----------------+--------------------+
| id|              title|is_overview_available|is_genre_available|is_director_available|is_cast_available|is_keyword_available|
+---+-------------------+---------------------+------------------+---------------------+-----------------+--------------------+
|  3|Shadows in Paradise|                    1|                 1|                    1|                1|                   1|
|  5|         Four Rooms|                    1|                 1|                    1|                1|                   1|
|  6|     Judgment Night|                    1|                 1|                    1|                1|                   1|
|  9|   Sunday in August|                    1|                 1|                    1|                1|                   1|
| 12|       Finding Nemo|                    1|                 1|                    1|                

## 10. Membersihkan Teks

Kolom teks seperti overview, genre, cast, director, dan keyword dibersihkan agar lebih konsisten.  
Pembersihan yang dilakukan:

- mengubah teks menjadi huruf kecil,
- menghapus karakter yang tidak terlalu berguna,
- merapikan spasi berlebih,
- membuat versi teks bersih untuk kebutuhan model rekomendasi.

In [22]:
def clean_text_column(col_expr):
    return F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(F.lower(col_expr), r"[^a-z0-9\s\|\-]", " "),
                r"\s+", " "
            ),
            r"\s*\|\s*", " | "
        )
    )

cleanable_text_cols = [
    "title", "original_title", "overview", "tagline", "genres", "genres_detail",
    "production_companies", "production_countries", "spoken_languages",
    "director", "writers", "top_cast", "keywords"
]

for c in cleanable_text_cols:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c + "_clean", clean_text_column(F.col(c)))

df_clean.select("title", "title_clean", "overview_clean").show(3, truncate=80)

+-------------------+-------------------+--------------------------------------------------------------------------------+
|              title|        title_clean|                                                                  overview_clean|
+-------------------+-------------------+--------------------------------------------------------------------------------+
|Shadows in Paradise|shadows in paradise|nikander a rubbish collector and would-be entrepreneur finds his plans for su...|
|         Four Rooms|         four rooms|it s ted the bellhop s first night on the job and the hotel s very unusual gu...|
|     Judgment Night|     judgment night|four young friends while taking a shortcut en route to a local boxing match w...|
+-------------------+-------------------+--------------------------------------------------------------------------------+
only showing top 3 rows


## 11. Mengubah Kolom Multi-Value Menjadi Array

Beberapa kolom hasil enrichment memiliki banyak nilai dalam satu string, misalnya:

```text
Action | Drama | Thriller
```

Agar lebih mudah diproses, kolom tersebut diubah menjadi array:

```text
["action", "drama", "thriller"]
```

Kolom array ini berguna untuk analisis genre, cast, director, keyword, dan pembuatan dokumen rekomendasi.

In [23]:
def split_pipe_to_array(column_name: str):
    # Memakai SQL expression supaya bisa trim dan membuang elemen kosong
    return F.expr(
        f"filter(transform(split({column_name}, '\\\\s*\\\\|\\\\s*'), x -> trim(x)), x -> x is not null and x <> '')"
    )

multi_value_sources = {
    "genres_detail_clean": "genres_list",
    "genres_clean": "genres_raw_list",
    "production_companies_clean": "production_companies_list",
    "production_countries_clean": "production_countries_list",
    "spoken_languages_clean": "spoken_languages_list",
    "writers_clean": "writers_list",
    "top_cast_clean": "top_cast_list",
    "keywords_clean": "keywords_list",
}

for source_col, target_col in multi_value_sources.items():
    if source_col in df_clean.columns:
        df_clean = df_clean.withColumn(target_col, split_pipe_to_array(source_col))

# Jika genres_detail kosong tetapi genres tersedia, gunakan genres_raw_list
if "genres_list" in df_clean.columns and "genres_raw_list" in df_clean.columns:
    df_clean = df_clean.withColumn(
        "genres_final_list",
        F.when(F.size(F.col("genres_list")) > 0, F.col("genres_list"))
         .otherwise(F.col("genres_raw_list"))
    )
elif "genres_list" in df_clean.columns:
    df_clean = df_clean.withColumn("genres_final_list", F.col("genres_list"))
elif "genres_raw_list" in df_clean.columns:
    df_clean = df_clean.withColumn("genres_final_list", F.col("genres_raw_list"))

df_clean.select("title", "genres_final_list", "top_cast_list", "keywords_list").show(5, truncate=100)

+-------------------+------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|              title|             genres_final_list|                                                                                       top_cast_list|                                                                                       keywords_list|
+-------------------+------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|Shadows in Paradise|      [comedy, drama, romance]|[matti pellonp, kati outinen, sakari kuosmanen, esko nikkari, kylli k ng s, pekka laiho, jukka-pe...|                                                             [helsinki finland, sa

## 12. Feature Engineering Numerik

Tahap ini membuat fitur numerik tambahan agar data lebih siap digunakan pada model.  
Fitur yang dibuat:

- `log_popularity`,
- `log_vote_count`,
- `log_budget`,
- `log_revenue`,
- `log_runtime`,
- flag budget/revenue valid,
- kategori tahun/era film.

Transformasi `log1p` digunakan agar distribusi numerik yang sangat miring menjadi lebih stabil.

In [24]:
df_features = df_clean

# Tahun rilis jika belum ada
if "release_year" not in df_features.columns:
    date_col = "release_date_parsed_clean" if "release_date_parsed_clean" in df_features.columns else "release_date"
    df_features = df_features.withColumn("release_year", F.year(F.to_date(F.col(date_col))))

df_features = (
    df_features
    .withColumn("budget_valid", F.when(F.col("budget") > 0, 1).otherwise(0))
    .withColumn("revenue_valid", F.when(F.col("revenue") > 0, 1).otherwise(0))
    .withColumn("runtime_valid", F.when((F.col("runtime") >= 40) & (F.col("runtime") <= 240), 1).otherwise(0))
    .withColumn("vote_count_valid", F.when(F.col("vote_count") >= 10, 1).otherwise(0))
    .withColumn("log_popularity", F.log1p(F.col("popularity")))
    .withColumn("log_vote_count", F.log1p(F.col("vote_count")))
    .withColumn("log_budget", F.log1p(F.col("budget")))
    .withColumn("log_revenue", F.log1p(F.col("revenue")))
    .withColumn("log_runtime", F.log1p(F.col("runtime")))
)

df_features = df_features.withColumn(
    "movie_era",
    F.when(F.col("release_year") < 1990, "classic_1980s")
     .when((F.col("release_year") >= 1990) & (F.col("release_year") < 2000), "1990s")
     .when((F.col("release_year") >= 2000) & (F.col("release_year") < 2010), "2000s")
     .when((F.col("release_year") >= 2010) & (F.col("release_year") < 2020), "2010s")
     .when(F.col("release_year") >= 2020, "2020s")
     .otherwise("unknown")
)

df_features.select(
    "title", "release_year", "movie_era", "runtime", "log_runtime",
    "vote_count", "log_vote_count", "budget", "log_budget"
).show(5, truncate=80)

+-------------------+------------+-------------+-------+-----------------+----------+------------------+---------+------------------+
|              title|release_year|    movie_era|runtime|      log_runtime|vote_count|    log_vote_count|   budget|        log_budget|
+-------------------+------------+-------------+-------+-----------------+----------+------------------+---------+------------------+
|Shadows in Paradise|        1986|classic_1980s|   74.0| 4.31748811353631|       440| 6.089044875446846|      0.0|               0.0|
|         Four Rooms|        1995|        1990s|   98.0| 4.59511985013459|      2851| 7.955775781534187|4000000.0|15.201805169084134|
|     Judgment Night|        1993|        1990s|  109.0|4.700480365792417|       377| 5.934894195619588|    2.1E7|16.860033043306743|
|   Sunday in August|        2004|        2000s|   15.0|2.772588722239781|        30|3.4339872044851463|      0.0|               0.0|
|       Finding Nemo|        2003|        2000s|  100.0| 4.615

## 13. Quality Score untuk Data Rekomendasi

EDA sebelumnya sudah memiliki `metadata_completeness_score`.  
Pada tahap preprocessing ini dibuat skor tambahan yaitu `recommendation_quality_score`.

Skor ini memperhitungkan:

- overview tersedia,
- genre tersedia,
- director tersedia,
- cast tersedia,
- keywords tersedia,
- runtime valid,
- vote count valid.

Skor ini berguna untuk memilih data berkualitas tinggi saat training model rekomendasi.

In [25]:
df_features = df_features.withColumn(
    "recommendation_quality_score",
    (
        F.col("is_overview_available") +
        F.col("is_genre_available") +
        F.col("is_director_available") +
        F.col("is_cast_available") +
        F.col("is_keyword_available") +
        F.col("runtime_valid") +
        F.col("vote_count_valid")
    ) / F.lit(7.0)
)

df_features.select(
    "title", "recommendation_quality_score",
    "is_overview_available", "is_genre_available", "is_director_available",
    "is_cast_available", "is_keyword_available", "runtime_valid", "vote_count_valid"
).show(10, truncate=80)

df_features.select(
    F.round(F.avg("recommendation_quality_score"), 4).alias("avg_recommendation_quality_score"),
    F.round(F.min("recommendation_quality_score"), 4).alias("min_score"),
    F.round(F.max("recommendation_quality_score"), 4).alias("max_score")
).show()

+-------------------+----------------------------+---------------------+------------------+---------------------+-----------------+--------------------+-------------+----------------+
|              title|recommendation_quality_score|is_overview_available|is_genre_available|is_director_available|is_cast_available|is_keyword_available|runtime_valid|vote_count_valid|
+-------------------+----------------------------+---------------------+------------------+---------------------+-----------------+--------------------+-------------+----------------+
|Shadows in Paradise|                         1.0|                    1|                 1|                    1|                1|                   1|            1|               1|
|         Four Rooms|                         1.0|                    1|                 1|                    1|                1|                   1|            1|               1|
|     Judgment Night|                         1.0|                    1|        

## 14. Membuat Dokumen Film untuk Model Rekomendasi

Model rekomendasi berbasis konten membutuhkan representasi teks film.  
Pada bagian ini dibuat beberapa versi dokumen:

1. `movie_document_basic`  
   Berisi judul, genre, overview.

2. `movie_document_rich`  
   Berisi judul, genre, director, cast, keyword, tagline, overview.

3. `movie_document_weighted`  
   Beberapa metadata penting seperti judul, genre, cast, dan keyword diulang agar bobotnya lebih kuat saat diproses oleh TF-IDF atau embedding.

Kolom ini nanti bisa dipakai untuk:

- TF-IDF + Cosine Similarity,
- CountVectorizer,
- Sentence Embedding,
- Qdrant vector search.

In [26]:
# Ubah array menjadi teks
array_text_cols = {
    "genres_final_list": "genres_text",
    "top_cast_list": "top_cast_text",
    "keywords_list": "keywords_text",
    "writers_list": "writers_text",
    "production_companies_list": "production_companies_text",
    "production_countries_list": "production_countries_text",
    "spoken_languages_list": "spoken_languages_text",
}

for arr_col, text_col in array_text_cols.items():
    if arr_col in df_features.columns:
        df_features = df_features.withColumn(text_col, F.concat_ws(" ", F.col(arr_col)))
    else:
        df_features = df_features.withColumn(text_col, F.lit(""))

# Pastikan kolom teks bersih yang digunakan untuk dokumen film selalu tersedia
required_clean_text_defaults = [
    "title_clean",
    "original_title_clean",
    "overview_clean",
    "director_clean",
    "tagline_clean",
]

for c in required_clean_text_defaults:
    if c not in df_features.columns:
        base_col = c.replace("_clean", "")
        if base_col in df_features.columns:
            df_features = df_features.withColumn(c, clean_text_column(F.col(base_col)))
        else:
            df_features = df_features.withColumn(c, F.lit(""))

df_features = (
    df_features
    .withColumn(
        "movie_document_basic",
        F.trim(F.concat_ws(
            " ",
            F.col("title_clean"),
            F.col("genres_text"),
            F.col("overview_clean")
        ))
    )
    .withColumn(
        "movie_document_rich",
        F.trim(F.concat_ws(
            " ",
            F.col("title_clean"),
            F.col("original_title_clean"),
            F.col("genres_text"),
            F.col("director_clean"),
            F.col("writers_text"),
            F.col("top_cast_text"),
            F.col("keywords_text"),
            F.col("tagline_clean"),
            F.col("overview_clean")
        ))
    )
    .withColumn(
        "movie_document_weighted",
        F.trim(F.concat_ws(
            " ",
            F.col("title_clean"),
            F.col("title_clean"),
            F.col("genres_text"),
            F.col("genres_text"),
            F.col("director_clean"),
            F.col("top_cast_text"),
            F.col("top_cast_text"),
            F.col("keywords_text"),
            F.col("keywords_text"),
            F.col("overview_clean"),
            F.col("tagline_clean")
        ))
    )
    .withColumn("movie_document_length", F.length(F.col("movie_document_weighted")))
)

df_features.select("title", "movie_document_weighted").show(3, truncate=160)

+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|              title|                                                                                                                                         movie_document_weighted|
+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Shadows in Paradise|shadows in paradise shadows in paradise comedy drama romance comedy drama romance aki kaurism ki matti pellonp kati outinen sakari kuosmanen esko nikkari kyl...|
|         Four Rooms|four rooms four rooms comedy comedy allison anders | alexandre rockwell | quentin tarantino | robert rodriguez tim roth jennifer beals david proval ione skye...|
|     Judgment Night|judgment night judgment night action crime thriller action crime

## 15. Filtering Dataset Siap Model

Tahap ini membuat dataset yang lebih siap untuk model rekomendasi.  
Tidak semua film harus dibuang, tetapi untuk training awal sebaiknya gunakan film yang minimal memiliki:

- judul,
- genre,
- overview atau keyword/cast/director,
- vote count valid,
- kualitas metadata cukup.

Dataset ini akan disimpan sebagai `tmdb_movies_model_ready.csv`.

In [27]:
df_model_ready = (
    df_features
    .filter(F.col("movie_document_length") > 20)
    .filter(F.col("recommendation_quality_score") >= 0.45)
)

print("Rows semua fitur   :", df_features.count())
print("Rows model ready   :", df_model_ready.count())

df_model_ready.select(
    "id", "title", "release_year", "genres_text",
    "vote_average", "vote_count", "recommendation_quality_score"
).show(10, truncate=100)

Rows semua fitur   : 80712
Rows model ready   : 80549
+---+-------------------+------------+--------------------------------+------------+----------+----------------------------+
| id|              title|release_year|                     genres_text|vote_average|vote_count|recommendation_quality_score|
+---+-------------------+------------+--------------------------------+------------+----------+----------------------------+
|  3|Shadows in Paradise|        1986|            comedy drama romance|       7.258|       440|                         1.0|
|  5|         Four Rooms|        1995|                          comedy|       5.905|      2851|                         1.0|
|  6|     Judgment Night|        1993|           action crime thriller|       6.463|       377|                         1.0|
|  9|   Sunday in August|        2004|                           drama|       6.967|        30|          0.8571428571428571|
| 12|       Finding Nemo|        2003|      animation family adventure|

## 16. Membuat Dataset Khusus Dokumen Film

Dataset ini berisi kolom-kolom inti yang nanti bisa langsung digunakan untuk:

- TF-IDF,
- embedding,
- MongoDB document,
- Qdrant payload,
- sistem RAG menggunakan n8n.

Kolom yang disimpan dibuat lebih ringkas agar mudah dipakai pada tahap berikutnya.

In [28]:
document_columns = [
    "id",
    "title",
    "original_title",
    "release_date",
    "release_year",
    "movie_era",
    "original_language",
    "genres_text",
    "director_clean",
    "top_cast_text",
    "keywords_text",
    "overview_clean",
    "tagline_clean",
    "runtime",
    "vote_average",
    "vote_count",
    "popularity",
    "budget",
    "revenue",
    "recommendation_quality_score",
    "movie_document_basic",
    "movie_document_rich",
    "movie_document_weighted",
]

document_columns = [c for c in document_columns if c in df_model_ready.columns]

df_movie_documents = df_model_ready.select(*document_columns)

print("Kolom movie document:")
print(df_movie_documents.columns)

df_movie_documents.show(5, truncate=100)

Kolom movie document:
['id', 'title', 'original_title', 'release_date', 'release_year', 'movie_era', 'original_language', 'genres_text', 'director_clean', 'top_cast_text', 'keywords_text', 'overview_clean', 'tagline_clean', 'runtime', 'vote_average', 'vote_count', 'popularity', 'budget', 'revenue', 'recommendation_quality_score', 'movie_document_basic', 'movie_document_rich', 'movie_document_weighted']
+---+-------------------+--------------------+------------+------------+-------------+-----------------+--------------------------+--------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------

## 17. Preprocessing Opsional dengan Spark ML: Tokenisasi, Stopword Removal, TF-IDF

Bagian ini bersifat opsional, tetapi berguna untuk menunjukkan bahwa dataset sudah siap masuk ke pipeline machine learning.  
Tahap yang dilakukan:

1. tokenisasi dokumen film,
2. hapus stopwords bahasa Inggris,
3. bentuk representasi TF,
4. bentuk representasi TF-IDF.

Output vektor tidak disimpan ke CSV karena format vector Spark tidak cocok untuk CSV biasa.  
Bagian ini cukup digunakan sebagai validasi bahwa teks hasil preprocessing siap dipakai untuk model.

In [29]:
# Ambil subset untuk validasi pipeline agar cepat
df_tfidf_sample = df_movie_documents.select("id", "title", "movie_document_weighted").limit(10000)

tokenizer = RegexTokenizer(
    inputCol="movie_document_weighted",
    outputCol="tokens",
    pattern="\\W+",
    minTokenLength=2
)

tokenized = tokenizer.transform(df_tfidf_sample)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

filtered = remover.transform(tokenized)

cv = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="tf_features",
    vocabSize=20000,
    minDF=3.0
)

cv_model = cv.fit(filtered)
tf_df = cv_model.transform(filtered)

idf = IDF(
    inputCol="tf_features",
    outputCol="tfidf_features"
)

idf_model = idf.fit(tf_df)
tfidf_df = idf_model.transform(tf_df)

print("Contoh vocabulary:", cv_model.vocabulary[:30])
tfidf_df.select("id", "title", "filtered_tokens", "tfidf_features").show(5, truncate=100)

Contoh vocabulary: ['drama', 'comedy', 'thriller', 'action', 'romance', 'family', 'horror', 'john', 'crime', 'michael', 'love', 'life', 'woman', 'based', 'one', 'david', 'new', 'man', 'adventure', 'relationship', 'war', 'movie', 'young', 'world', 'fiction', 'science', 'james', 'murder', 'robert', 'story']
+---+-------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
| id|              title|                                                                                     filtered_tokens|                                                                                      tfidf_features|
+---+-------------------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|  3|Shad

## 18. Ringkasan Kualitas Setelah Preprocessing

Bagian ini membuat ringkasan hasil preprocessing yang bisa dipakai pada laporan atau presentasi.  
Ringkasan meliputi:

- jumlah data awal dan data siap model,
- rata-rata kualitas metadata,
- jumlah film per era,
- ketersediaan genre/director/cast/keywords,
- distribusi vote count dan rating.

In [30]:
from pyspark.sql.types import StructType, StructField, StringType

# Ringkasan preprocessing.
# Cell ini TIDAK membuat ulang df_features, karena df_features sudah dibuat pada tahap feature engineering.
summary_rows = []

# Jumlah data
total_rows = df_features.count()
model_ready_rows = df_model_ready.count() if "df_model_ready" in globals() else 0
document_rows = df_movie_documents.count() if "df_movie_documents" in globals() else 0

summary_rows.append(("total_rows_preprocessed_full", total_rows))
summary_rows.append(("total_rows_model_ready", model_ready_rows))
summary_rows.append(("total_rows_movie_documents", document_rows))
summary_rows.append(("total_columns_preprocessed_full", len(df_features.columns)))

# Deduplikasi berdasarkan id
if "id" in df_features.columns:
    unique_movies = df_features.select("id").distinct().count()
    duplicate_movies = total_rows - unique_movies
    summary_rows.append(("unique_movies", unique_movies))
    summary_rows.append(("duplicate_movies", duplicate_movies))

# Kolom numerik penting
numeric_summary_cols = [
    "vote_average",
    "vote_count",
    "popularity",
    "runtime",
    "budget",
    "revenue",
    "profit",
    "roi",
    "metadata_completeness_score",
    "recommendation_quality_score",
    "movie_document_length",
]

for c in numeric_summary_cols:
    if c in df_features.columns:
        row = df_features.select(
            F.round(F.avg(F.col(c)), 4).alias("avg_value"),
            F.round(F.min(F.col(c)), 4).alias("min_value"),
            F.round(F.max(F.col(c)), 4).alias("max_value"),
        ).collect()[0]

        summary_rows.append((f"avg_{c}", row["avg_value"]))
        summary_rows.append((f"min_{c}", row["min_value"]))
        summary_rows.append((f"max_{c}", row["max_value"]))

# Flag kelengkapan metadata
flag_cols = [
    "is_overview_available",
    "is_genre_available",
    "is_director_available",
    "is_cast_available",
    "is_keyword_available",
    "budget_valid",
    "revenue_valid",
    "runtime_valid",
    "vote_count_valid",
]

for c in flag_cols:
    if c in df_features.columns:
        value = df_features.select(F.round(F.avg(F.col(c)), 4).alias("rate_value")).collect()[0]["rate_value"]
        summary_rows.append((f"rate_{c}", value))

# Simpan sebagai string agar tidak bentrok tipe LongType dan DoubleType
summary_rows_clean = [(str(metric), "" if value is None else str(value)) for metric, value in summary_rows]

summary_schema = StructType([
    StructField("metric", StringType(), True),
    StructField("value", StringType(), True),
])

summary_df = spark.createDataFrame(summary_rows_clean, schema=summary_schema)

summary_df.show(100, truncate=False)

print("Distribusi era:")
if "movie_era" in df_features.columns:
    df_features.groupBy("movie_era").count().orderBy(F.desc("count")).show(truncate=False)

+--------------------------------+-------------+
|metric                          |value        |
+--------------------------------+-------------+
|total_rows_preprocessed_full    |80712        |
|total_rows_model_ready          |80549        |
|total_rows_movie_documents      |80549        |
|total_columns_preprocessed_full |125          |
|unique_movies                   |80712        |
|duplicate_movies                |0            |
|avg_vote_average                |6.0062       |
|min_vote_average                |1.2          |
|max_vote_average                |10.0         |
|avg_vote_count                  |302.1196     |
|min_vote_count                  |10           |
|max_vote_count                  |39973        |
|avg_popularity                  |1.2266       |
|min_popularity                  |0.0          |
|max_popularity                  |881.8242     |
|avg_runtime                     |94.0688      |
|min_runtime                     |0.0          |
|max_runtime        

## 19. Export Dataset Hasil Preprocessing

Karena Spark lokal di Windows bisa bermasalah saat menulis Parquet/CSV menggunakan Hadoop writer, export akhir dilakukan dengan Pandas.  
Proses transformasi tetap dilakukan menggunakan PySpark, sedangkan Pandas hanya digunakan untuk menyimpan file akhir.

Output yang dibuat:

1. `tmdb_movies_preprocessed_full.csv`  
   Dataset lengkap dengan fitur hasil preprocessing.

2. `tmdb_movies_model_ready.csv`  
   Dataset yang sudah difilter dan siap untuk model.

3. `tmdb_movie_documents.csv`  
   Dataset dokumen film untuk TF-IDF, embedding, MongoDB, Qdrant, atau n8n.

4. `preprocessing_summary.csv`  
   Ringkasan metrik preprocessing.

In [31]:
FULL_OUTPUT = PROCESSED_DIR / "tmdb_movies_preprocessed_full.csv"
MODEL_READY_OUTPUT = PROCESSED_DIR / "tmdb_movies_model_ready.csv"
DOCUMENT_OUTPUT = PROCESSED_DIR / "tmdb_movie_documents.csv"
SUMMARY_OUTPUT = REPORT_DIR / "preprocessing_summary.csv"

# Untuk CSV, array column sebaiknya diubah ke string
array_cols_to_export = [
    "genres_final_list", "genres_list", "genres_raw_list",
    "production_companies_list", "production_countries_list",
    "spoken_languages_list", "writers_list", "top_cast_list", "keywords_list"
]

df_features_export = df_features
for c in array_cols_to_export:
    if c in df_features_export.columns:
        df_features_export = df_features_export.withColumn(c + "_str", F.concat_ws(" | ", F.col(c))).drop(c)

df_model_ready_export = df_model_ready
for c in array_cols_to_export:
    if c in df_model_ready_export.columns:
        df_model_ready_export = df_model_ready_export.withColumn(c + "_str", F.concat_ws(" | ", F.col(c))).drop(c)

# Export memakai Pandas agar aman di Windows
pdf_full = df_features_export.toPandas()
pdf_model_ready = df_model_ready_export.toPandas()
pdf_documents = df_movie_documents.toPandas()
pdf_summary = summary_df.toPandas()

pdf_full.to_csv(FULL_OUTPUT, index=False)
pdf_model_ready.to_csv(MODEL_READY_OUTPUT, index=False)
pdf_documents.to_csv(DOCUMENT_OUTPUT, index=False)
pdf_summary.to_csv(SUMMARY_OUTPUT, index=False)

print("Saved full       :", FULL_OUTPUT, "rows:", len(pdf_full))
print("Saved model ready:", MODEL_READY_OUTPUT, "rows:", len(pdf_model_ready))
print("Saved documents  :", DOCUMENT_OUTPUT, "rows:", len(pdf_documents))
print("Saved summary    :", SUMMARY_OUTPUT, "rows:", len(pdf_summary))

Saved full       : d:\Semester 6\BigData\movie-recommendation-bigdata\data\processed\tmdb\tmdb_movies_preprocessed_full.csv rows: 80712
Saved model ready: d:\Semester 6\BigData\movie-recommendation-bigdata\data\processed\tmdb\tmdb_movies_model_ready.csv rows: 80549
Saved documents  : d:\Semester 6\BigData\movie-recommendation-bigdata\data\processed\tmdb\tmdb_movie_documents.csv rows: 80549
Saved summary    : d:\Semester 6\BigData\movie-recommendation-bigdata\reports\preprocessing\preprocessing_summary.csv rows: 48


## 20. Validasi File Output

Tahap akhir ini membaca ulang file CSV hasil preprocessing menggunakan Pandas untuk memastikan file benar-benar tersimpan dan jumlah datanya sesuai.

In [32]:
import pandas as pd

check_full = pd.read_csv(FULL_OUTPUT, nrows=5)
check_model = pd.read_csv(MODEL_READY_OUTPUT, nrows=5)
check_doc = pd.read_csv(DOCUMENT_OUTPUT, nrows=5)

print("Full output columns:", len(check_full.columns))
print("Model ready columns:", len(check_model.columns))
print("Document columns:", len(check_doc.columns))

display(check_doc.head())

Full output columns: 125
Model ready columns: 125
Document columns: 23


,id,title,original_title,release_date,release_year,movie_era,original_language,genres_text,director_clean,top_cast_text,...,runtime,vote_average,vote_count,popularity,budget,revenue,recommendation_quality_score,movie_document_basic,movie_document_rich,movie_document_weighted
0,3,Shadows in Paradise,Varjoja paratiisissa,1986-10-17,1986,classic_1980s,fi,comedy drama romance,aki kaurism ki,matti pellonp kati outinen sakari kuosmanen es...,...,74.0,7.258,440,1.5588,0.0,0.0,1.000000,shadows in paradise comedy drama romance nikan...,shadows in paradise varjoja paratiisissa comed...,shadows in paradise shadows in paradise comedy...
1,5,Four Rooms,Four Rooms,1995-12-09,1995,1990s,en,comedy,allison anders | alexandre rockwell | quentin ...,tim roth jennifer beals david proval ione skye...,...,98.0,5.905,2851,2.9909,4000000.0,4257354.0,1.000000,four rooms comedy it s ted the bellhop s first...,four rooms four rooms comedy allison anders | ...,four rooms four rooms comedy comedy allison an...
2,6,Judgment Night,Judgment Night,1993-10-15,1993,1990s,en,action crime thriller,stephen hopkins,emilio estevez cuba gooding jr denis leary ste...,...,109.0,6.463,377,1.4890,21000000.0,12136938.0,1.000000,judgment night action crime thriller four youn...,judgment night judgment night action crime thr...,judgment night judgment night action crime thr...
3,9,Sunday in August,"Sonntag, im August",2004-09-02,2004,2000s,de,drama,marc meyer,rita lengyel milton welsh,...,15.0,6.967,30,0.3611,0.0,0.0,0.857143,sunday in august drama a couple on a boat thei...,sunday in august sonntag im august drama marc ...,sunday in august sunday in august drama drama ...
4,12,Finding Nemo,Finding Nemo,2003-05-30,2003,2000s,en,animation family adventure,andrew stanton,albert brooks ellen degeneres alexander gould ...,...,100.0,7.819,20581,19.3039,94000000.0,940335536.0,1.000000,finding nemo animation family adventure nemo a...,finding nemo finding nemo animation family adv...,finding nemo finding nemo animation family adv...


## 21. Catatan untuk Tahap Berikutnya

Setelah preprocessing selesai, output yang paling penting adalah:

```text
data/processed/tmdb/tmdb_movies_model_ready.csv
data/processed/tmdb/tmdb_movie_documents.csv
```

Rekomendasi penggunaan:

- `tmdb_movies_model_ready.csv` digunakan untuk training model rekomendasi berbasis metadata.
- `tmdb_movie_documents.csv` digunakan untuk TF-IDF, embedding, Qdrant, MongoDB, dan RAG.
- `preprocessing_summary.csv` digunakan untuk laporan atau presentasi.

Tahap berikutnya bisa dilanjutkan ke:

1. training TF-IDF + Cosine Similarity,
2. training model clustering,
3. pembuatan embedding,
4. import metadata ke MongoDB,
5. insert vector ke Qdrant.

In [33]:
# Jalankan cell ini jika sudah selesai bekerja dengan Spark.
# spark.stop()